# Regressão Logística com Regularização

Neste notebook, você irá implementar a regressão logística **do zero** com regularização **L1** e **L2**, utilizando gradiente descendente.

In [5]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from tqdm import tqdm
np.random.seed(42)

## Carregar o conjunto de dados

In [6]:
breast_cancer = load_breast_cancer()
X = breast_cancer.data
y = breast_cancer.target
pd.DataFrame(X, columns=breast_cancer.feature_names)

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,...,25.380,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,24.990,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,...,23.570,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,...,14.910,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,...,22.540,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,0.05623,...,25.450,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115
565,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,0.05533,...,23.690,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637
566,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,0.05648,...,18.980,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820
567,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,0.07016,...,25.740,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400


## Pré-processamento

In [7]:
# Divida os dados em treino e teste (test_size=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Treino:", X_train.shape)
print("Teste :", X_test.shape)

Treino: (455, 30)
Teste : (114, 30)


In [8]:
# Normalize os dados com StandardScaler
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [9]:
# Adicione a coluna de bias (coluna de 1s) ao início de X_train e X_test
X_train = np.c_[np.ones(X_train.shape[0]), X_train]
X_test = np.c_[np.ones(X_test.shape[0]), X_test]

print(X_train.shape)
print(X_test.shape)

(455, 31)
(114, 31)


## Função de Custo com Regularização

A função de custo da regressão logística com regularização é:

$$J(w) = -\frac{1}{m}\sum_{i=1}^{m}\left[y^{(i)}\log(\hat{y}^{(i)}) + (1-y^{(i)})\log(1-\hat{y}^{(i)})\right] + R(w)$$

Onde $R(w)$ é o termo de regularização:
- **L2 (Ridge):** $\lambda \sum_{j=1}^{n} w_j^2$
- **L1 (Lasso):** $\lambda \sum_{j=1}^{n} |w_j|$

> **Importante:** O termo de bias ($w_0$) **não** deve ser regularizado!

In [10]:
# Implemente a função de custo com regularização
# Dica: use np.clip para evitar log(0)
# Dica: não regularize o bias (weights[0])

def cross_entropy_with_regularization(
    y_true,
    y_pred,
    weights,
    lbda,
    regularization='l2'
):

    m = len(y_true)

    # Evita log(0)
    y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)

    # Cross-Entropy
    cost = (
        -1 / m
        * np.sum(
            y_true * np.log(y_pred)
            + (1 - y_true) * np.log(1 - y_pred)
        )
    )




    w = weights[1:]

  
    if regularization == 'l2':

        reg_term = (lbda / (2 * m)) * np.sum(w ** 2)


    elif regularization == 'l1':

        reg_term = (lbda / m) * np.sum(np.abs(w))

    else:

        reg_term = 0


    total_cost = cost + reg_term

    return total_cost

In [11]:
# teste a função de custo
w_test = np.array([0.5, 0.3, -0.2, 0.8])
y_test_true = np.array([1, 0, 1, 1])
y_test_pred = np.array([0.9, 0.1, 0.8, 0.7])
print("L2:", cross_entropy_with_regularization(y_test_true, y_test_pred, w_test, 0.01, 'l2'))
print("L1:", cross_entropy_with_regularization(y_test_true, y_test_pred, w_test, 0.01, 'l1'))

L2: 0.19859738164214868
L1: 0.20088488164214868


## Função Sigmoide

In [12]:
# Implemente a função sigmoide
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

## Treinamento com Gradiente Descendente

O gradiente da função de custo com regularização em relação aos pesos é:

$$\frac{\partial J}{\partial w} = \frac{1}{m} X^T(\hat{y} - y) + \frac{\partial R(w)}{\partial w}$$

Gradientes dos termos de regularização (não aplicar ao bias):
- **L2:** $2\lambda w_j$
- **L1:** $\lambda \cdot \text{sign}(w_j)$  

Implemente o treinamento. Selecione o tipo de regularização definindo a variável `regularization` como `None`, `'l1'` ou `'l2'` antes do loop.

In [13]:
# Selecione o tipo de regularização: None, 'l1' ou 'l2'
# Use: lr = 0.001, epochs = 200, lbda = 0.01
# Dica: não aplique o gradiente de regularização ao bias (weights[0])

regularization = 'l2'

lr = 0.001
epochs = 200
lbda = 0.01

m, n = X_train.shape

weights = np.zeros(n)

cost_history = []

for epoch in range(epochs):

    z = X_train @ weights

    y_pred = sigmoid(z)

    error = y_pred - y_train

    gradient = (1 / m) * (X_train.T @ error)

    if regularization == 'l2':

        reg_gradient = (lbda / m) * weights

        reg_gradient[0] = 0

        gradient += reg_gradient

    elif regularization == 'l1':

        reg_gradient = (lbda / m) * np.sign(weights)

        reg_gradient[0] = 0

        gradient += reg_gradient

    weights -= lr * gradient

    cost = cross_entropy_with_regularization(
        y_train,
        y_pred,
        weights,
        lbda,
        regularization
    )

    cost_history.append(cost)

    if epoch % 20 == 0:

        print(f"Epoch {epoch} | Cost: {cost:.6f}")

Epoch 0 | Cost: 0.693147
Epoch 20 | Cost: 0.656401
Epoch 40 | Cost: 0.623940
Epoch 60 | Cost: 0.595172
Epoch 80 | Cost: 0.569566
Epoch 100 | Cost: 0.546664
Epoch 120 | Cost: 0.526077
Epoch 140 | Cost: 0.507480
Epoch 160 | Cost: 0.490600
Epoch 180 | Cost: 0.475212


## Avaliação do Modelo

In [14]:
# Calcule e imprima: Accuracy, Precision, Recall e F1
# Use threshold de 0.5 para converter probabilidades em classes

y_pred = sigmoid(X_test @ weights)

y_pred = (y_pred >= 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(y_test, y_pred)

recall = recall_score(y_test, y_pred)

f1 = f1_score(y_test, y_pred)

print("Accuracy :", accuracy)

print("Precision:", precision)

print("Recall   :", recall)

print("F1 Score :", f1)

print("\nConfusion Matrix:\n")

print(confusion_matrix(y_test, y_pred))

Accuracy : 0.9649122807017544
Precision: 0.971830985915493
Recall   : 0.971830985915493
F1 Score : 0.971830985915493

Confusion Matrix:

[[41  2]
 [ 2 69]]
